# Week 7 Notebook 
- overall data quality checking
- follow up to feature engineering notebook

plan
remove lat long missings
calc iqr ranges (for important columns)
remove those outliers

# Setup


In [32]:
# imports
import pandas as pd 


In [33]:
# read-in

sold_df = pd.read_csv("../data/enriched/geo_enhanced_sold_data.csv", low_memory=False)
listings_df = pd.read_csv("../data/enriched/geo_enhanced_listings_data.csv", low_memory=False)

In [34]:
listings_df.head()

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,...,placeholder_coords_flag,non_cali_coords_flag,price_ratio,price_per_sqft,days_on_market,yr_month,listing_to_contract_days,contract_to_close_days,index_right,DistrictNa
0,929000.0,1076194146,NaN,NaN,NaN,NaN,16882 Canyon Lane,Residential,1389.0,929000.0,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,999999.0,1076194026,NaN,NaN,NaN,NaN,8720 S 4th Avenue,Residential,2526.0,999999.0,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1400000.0,1076193814,NaN,NaN,33.858559,-116.542169,505 E Molino Road,Residential,2256.0,1400000.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,478.0,Palm Springs Unified
3,4998888.0,1076193812,NaN,NaN,NaN,NaN,3653 Halldale Avenue,ResidentialIncome,NaN,4998888.0,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,549000.0,1076193525,NaN,NaN,NaN,NaN,1736 N Mcdivitt Avenue,Residential,986.0,549000.0,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
listings_df_edit = listings_df.dropna(subset=['Latitude', 'Longitude'])

In [36]:
listings_df_edit.info()

<class 'pandas.core.frame.DataFrame'>
Index: 941336 entries, 2 to 1053810
Data columns (total 67 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   OriginalListPrice            937525 non-null  float64
 1   ListingKey                   941336 non-null  int64  
 2   CloseDate                    277873 non-null  object 
 3   ClosePrice                   254176 non-null  float64
 4   Latitude                     941336 non-null  float64
 5   Longitude                    941336 non-null  float64
 6   UnparsedAddress              938962 non-null  object 
 7   PropertyType                 941336 non-null  object 
 8   LivingArea                   823986 non-null  float64
 9   ListPrice                    938870 non-null  float64
 10  DaysOnMarket                 941336 non-null  int64  
 11  ListOfficeName               941336 non-null  object 
 12  BuyerOfficeName              264463 non-null  object 
 13  CoL

In [37]:
def iqr_outlier_flag(df, subset, multiplier= 1.5, remove=False):
    Q1 = df[subset].quantile(0.25)
    Q3 = df[subset].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    if remove:
        df = df[(df[subset] >= lower) & (df[subset] <= upper)]
    else:
        df['outlier_flag'] = (df[subset] < lower) | (df[subset] > upper)
    return df
    

In [38]:
listings_df_edit = iqr_outlier_flag(listings_df_edit, subset='OriginalListPrice', multiplier=1.5, remove=True)
listings_df_edit.info()

<class 'pandas.core.frame.DataFrame'>
Index: 877633 entries, 2 to 1053810
Data columns (total 67 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   OriginalListPrice            877633 non-null  float64
 1   ListingKey                   877633 non-null  int64  
 2   CloseDate                    263689 non-null  object 
 3   ClosePrice                   243203 non-null  float64
 4   Latitude                     877633 non-null  float64
 5   Longitude                    877633 non-null  float64
 6   UnparsedAddress              875469 non-null  object 
 7   PropertyType                 877633 non-null  object 
 8   LivingArea                   772786 non-null  float64
 9   ListPrice                    877612 non-null  float64
 10  DaysOnMarket                 877633 non-null  int64  
 11  ListOfficeName               877633 non-null  object 
 12  BuyerOfficeName              253199 non-null  object 
 13  CoL

# Full Data Cleaning / Imputation